# Tweeting Fear — GPT-2 Fine-Tuning
### BUSN 20800 Big Data
**Hewitt Watkins · Erik Lopez · Mateo Fretes · Vedant Dangayach**

**Input:** `booth_final_data/tweet_labels.csv` from Google Drive  
**Output:** Fine-tuned model + results saved to `booth_results/` in Google Drive

> Run on a GPU runtime: Runtime → Change runtime type → T4 GPU

## 0. Mount Drive & Create Output Folders

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR    = '/content/drive/MyDrive/booth_final_data'
RESULTS_DIR = '/content/drive/MyDrive/booth_results'
MODELS_DIR  = os.path.join(RESULTS_DIR, 'models', 'trump_gpt2')
RESULTS_OUT = os.path.join(RESULTS_DIR, 'results')

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_OUT, exist_ok=True)

print('Drive mounted. Folders ready:')
print(f'  Data in:     {DATA_DIR}')
print(f'  Models out:  {MODELS_DIR}')
print(f'  Results out: {RESULTS_OUT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Folders ready:
  Data in:     /content/drive/MyDrive/booth_final_data
  Models out:  /content/drive/MyDrive/booth_results/models/trump_gpt2
  Results out: /content/drive/MyDrive/booth_results/results


In [2]:
import zipfile

zip_path    = os.path.join(DATA_DIR, 'data.zip')
labels_path = os.path.join(DATA_DIR, 'tweet_labels.csv')

# 1. Already at root level — nothing to do
if os.path.exists(labels_path):
    print(f'tweet_labels.csv found at root — ready.')

# 2. Already extracted into a data/ subfolder — update DATA_DIR so all cells below still work
elif os.path.exists(os.path.join(DATA_DIR, 'data', 'tweet_labels.csv')):
    DATA_DIR    = os.path.join(DATA_DIR, 'data')
    labels_path = os.path.join(DATA_DIR, 'tweet_labels.csv')
    print(f'tweet_labels.csv found in data/ subfolder — updated DATA_DIR to {DATA_DIR}')

# 3. Need to unzip
else:
    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f'Neither tweet_labels.csv nor data.zip found in {DATA_DIR}\n'
            'Please upload data.zip to booth_final_data/ in your Drive.'
        )
    print(f'Unzipping {zip_path} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print(f'Done. Contents of {DATA_DIR}:')
    for name in sorted(os.listdir(DATA_DIR)):
        size = os.path.getsize(os.path.join(DATA_DIR, name))
        print(f'  {name:40s}  {size/1e6:.1f} MB')
    # Re-check both locations after extraction
    if os.path.exists(os.path.join(DATA_DIR, 'tweet_labels.csv')):
        labels_path = os.path.join(DATA_DIR, 'tweet_labels.csv')
    elif os.path.exists(os.path.join(DATA_DIR, 'data', 'tweet_labels.csv')):
        DATA_DIR    = os.path.join(DATA_DIR, 'data')
        labels_path = os.path.join(DATA_DIR, 'tweet_labels.csv')
        print(f'Found in data/ subfolder after unzip — updated DATA_DIR to {DATA_DIR}')
    else:
        raise FileNotFoundError(
            'tweet_labels.csv not found after unzip. '
            'Check zip contents with: zipfile.ZipFile(zip_path).namelist()'
        )

print(f'Using tweet_labels.csv at: {labels_path}')


tweet_labels.csv found in data/ subfolder — updated DATA_DIR to /content/drive/MyDrive/booth_final_data/data
Using tweet_labels.csv at: /content/drive/MyDrive/booth_final_data/data/tweet_labels.csv


## 1. Install & Import

In [3]:
!pip install transformers datasets accelerate -q
print('Done')

Done


In [4]:
import pandas as pd
import numpy as np
import torch
import json
import random
import math
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset as HFDataset

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow. Change runtime to T4.')

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Memory: 102.0 GB


## 2. Load Data & Inspect Label Distribution

In [5]:
df = pd.read_csv(os.path.join(DATA_DIR, 'tweet_labels.csv'))
df = df.dropna(subset=['gpt2_prompt', 'vix_bucket', 'epu_bucket', 'lda_topic'])
df['lda_topic'] = df['lda_topic'].astype(int)
print(f'Loaded {len(df):,} labeled tweets')
print(f'Columns: {list(df.columns)}')
print()

# Label distribution
print('VIX bucket counts:')
print(df['vix_bucket'].value_counts().to_string())
print()
print('EPU bucket counts:')
print(df['epu_bucket'].value_counts().to_string())
print()
print('Topic counts:')
print(df['lda_topic'].value_counts().sort_index().to_string())
print()

# VIX x EPU x Topic combination counts
combo = df.groupby(['vix_bucket','epu_bucket','lda_topic']).size().reset_index(name='count')
print(f'Total conditioning combinations: {len(combo)}')
print(f'Sparsest combination: {combo["count"].min()} tweets')
print(f'Richest combination:  {combo["count"].max()} tweets')

Loaded 41,502 labeled tweets
Columns: ['tweet_idx', 'date', 'text', 'week_end', 'lda_topic', 'kmeans_cluster', 'vix_bucket', 'epu_bucket', 'vix', 'epu', 'gpt2_prompt']

VIX bucket counts:
vix_bucket
VIX_MED     14939
VIX_HIGH    14112
VIX_LOW     12451

EPU bucket counts:
epu_bucket
EPU_LOW     14989
EPU_HIGH    14575
EPU_MED     11938

Topic counts:
lda_topic
0    16612
1     7875
2     4647
3    12368

Total conditioning combinations: 36
Sparsest combination: 7 tweets
Richest combination:  4549 tweets


In [6]:
# Compute and save VIX/EPU bucket cutoffs from weekly data
# Use unique weekly values to match unsupervised notebook logic
weekly = df.groupby('week_end')[['vix','epu']].first().dropna()
vix_cuts = weekly['vix'].quantile([1/3, 2/3]).values
epu_cuts = weekly['epu'].quantile([1/3, 2/3]).values

cutoffs = {'vix': vix_cuts.tolist(), 'epu': epu_cuts.tolist()}
with open(os.path.join(RESULTS_OUT, 'bucket_cutoffs.json'), 'w') as f:
    json.dump(cutoffs, f, indent=2)

print(f'VIX tertile cutoffs:  {vix_cuts.round(2)}')
print(f'EPU tertile cutoffs:  {epu_cuts.round(2)}')
print('Saved bucket_cutoffs.json')

VIX tertile cutoffs:  [13.83 19.32]
EPU tertile cutoffs:  [119.78 163.24]
Saved bucket_cutoffs.json


## 3. Tokenizer & Model Setup

Add special conditioning tokens, resize model embeddings.

In [7]:
MODEL_NAME = 'gpt2'
tokenizer  = GPT2Tokenizer.from_pretrained(MODEL_NAME)

# Detect topics from data
topics = sorted(df['lda_topic'].unique())
print(f'Topics found: {topics}')

# Add all special conditioning tokens
special_tokens = [
    '[VIX_LOW]', '[VIX_MED]', '[VIX_HIGH]',
    '[EPU_LOW]', '[EPU_MED]', '[EPU_HIGH]',
] + [f'[TOPIC_{i}]' for i in topics]

tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
tokenizer.pad_token = tokenizer.eos_token

print(f'Vocabulary size after special tokens: {len(tokenizer):,}')

# Load model and resize embedding layer
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {params/1e6:.1f}M')
print('Model ready.')

Topics found: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Vocabulary size after special tokens: 50,267


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model parameters: 124.4M
Model ready.


## 4. Prepare Dataset

90/10 random train/test split. Format: `[VIX_X] [EPU_X] [TOPIC_N] <tweet text><|endoftext|>`

In [8]:
# Append EOS token to each prompt so model learns when to stop
df['training_text'] = df['gpt2_prompt'].astype(str) + tokenizer.eos_token

# 90/10 random split
random.seed(42)
indices   = list(range(len(df)))
random.shuffle(indices)
split_idx = int(0.9 * len(indices))
train_idx = indices[:split_idx]
test_idx  = indices[split_idx:]

train_texts = df.iloc[train_idx]['training_text'].tolist()
test_texts  = df.iloc[test_idx]['training_text'].tolist()
print(f'Train: {len(train_texts):,}  |  Test: {len(test_texts):,}')

def tokenize_fn(examples):
    # Do NOT set labels here — DataCollatorForLanguageModeling(mlm=False)
    # pads input_ids in each batch first, then copies to labels.
    # Pre-setting labels with variable-length lists causes a shape mismatch.
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=128,
    )

train_ds = HFDataset.from_dict({'text': train_texts}).map(
    tokenize_fn, batched=True, remove_columns=['text'])
test_ds  = HFDataset.from_dict({'text': test_texts}).map(
    tokenize_fn, batched=True, remove_columns=['text'])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print('Datasets ready.')
print(f'Train features: {train_ds.features}')


Train: 37,351  |  Test: 4,151


Map:   0%|          | 0/37351 [00:00<?, ? examples/s]

Map:   0%|          | 0/4151 [00:00<?, ? examples/s]

Datasets ready.
Train features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


## 5. Fine-Tune

3 epochs, checkpoints saved to Drive after each epoch.

In [9]:
training_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, 'models', 'checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=5e-5,
    fp16=(device.type == 'cuda'),
    logging_dir=os.path.join(RESULTS_OUT, 'logs'),
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=2,
    report_to='none',
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
)

print('Starting training...')
train_result = trainer.train()
print('Training complete.')
print(f'  Total steps:     {train_result.global_step}')
print(f'  Train loss:      {train_result.training_loss:.4f}')

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.606638,2.519616
2,2.399749,2.449684
3,2.257862,2.434844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training complete.
  Total steps:     14007
  Train loss:      2.5187


## 6. Save Model & Tokenizer to Drive

In [11]:
model.save_pretrained(MODELS_DIR)
tokenizer.save_pretrained(MODELS_DIR)
print(f'Model saved to {MODELS_DIR}')

# Save training metrics — cast numpy types to native Python for JSON
metrics = {
    'train_loss':    float(train_result.training_loss),
    'global_steps':  int(train_result.global_step),
    'train_samples': int(len(train_texts)),
    'test_samples':  int(len(test_texts)),
    'topics':        [int(t) for t in topics],
}
with open(os.path.join(RESULTS_OUT, 'training_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved training_metrics.json')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/booth_results/models/trump_gpt2
Saved training_metrics.json


## 7. Evaluate — Perplexity on Test Set

In [12]:
eval_results = trainer.evaluate()
perplexity   = math.exp(eval_results['eval_loss'])
print(f'Eval loss:   {eval_results["eval_loss"]:.4f}')
print(f'Perplexity:  {perplexity:.2f}')

with open(os.path.join(RESULTS_OUT, 'perplexity.txt'), 'w') as f:
    f.write(f'eval_loss:  {eval_results["eval_loss"]:.4f}\n')
    f.write(f'perplexity: {perplexity:.2f}\n')
print('Saved perplexity.txt')

Eval loss:   2.4348
Perplexity:  11.41
Saved perplexity.txt


## 8. Generate Sample Tweets (B2 — All Topics per VIX×EPU)

For every VIX×EPU combination, generate one tweet per topic. Saves full grid to Drive.

In [13]:
model.eval()

def generate_tweet(vix_label, epu_label, topic_id,
                   max_new_tokens=80, temperature=0.85, top_p=0.92):
    prompt    = f'[{vix_label}] [{epu_label}] [TOPIC_{topic_id}]'
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

VIX_LABELS = ['VIX_LOW', 'VIX_MED', 'VIX_HIGH']
EPU_LABELS = ['EPU_LOW', 'EPU_MED', 'EPU_HIGH']

sample_rows = []
print('Generating sample tweets for all VIX x EPU x Topic combinations...')
for vl in VIX_LABELS:
    for el in EPU_LABELS:
        for t in topics:
            text = generate_tweet(vl, el, t)
            sample_rows.append({
                'vix_label': vl, 'epu_label': el,
                'topic': t, 'generated_tweet': text
            })
        print(f'  {vl} x {el}: done')

samples_df = pd.DataFrame(sample_rows)
samples_df.to_csv(os.path.join(RESULTS_OUT, 'sample_generations.csv'), index=False)
print(f'Saved sample_generations.csv — {len(samples_df)} generated tweets')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generating sample tweets for all VIX x EPU x Topic combinations...
  VIX_LOW x EPU_LOW: done
  VIX_LOW x EPU_MED: done
  VIX_LOW x EPU_HIGH: done
  VIX_MED x EPU_LOW: done
  VIX_MED x EPU_MED: done
  VIX_MED x EPU_HIGH: done
  VIX_HIGH x EPU_LOW: done
  VIX_HIGH x EPU_MED: done
  VIX_HIGH x EPU_HIGH: done
Saved sample_generations.csv — 36 generated tweets


## 9. Preview Generated Tweets

In [14]:
for vl in VIX_LABELS:
    for el in EPU_LABELS:
        subset = samples_df[(samples_df['vix_label']==vl) & (samples_df['epu_label']==el)]
        print(f'\n{'='*60}')
        print(f'  [{vl}] [{el}]')
        print(f'{'='*60}')
        for _, row in subset.iterrows():
            print(f'  [Topic {row["topic"]}] {row["generated_tweet"][:140]}')


  [VIX_LOW] [EPU_LOW]
  [Topic 0] It was my great honor to welcome President @NarendraModi to the @WhiteHouse today! pic.twitter.com/GkPxRkqN2G7 — Donald J. Trump (@WhiteHous
  [Topic 1] “The Democrats had a field day. This is what we call an insurrection.” @TuckerCarlson Thank you @TulsiGabbard. @LouDobbs @FoxNews @seanhanni
  [Topic 2] Great to be with the Great Bill Barr and his wonderful wife, Laura. I know them both well! https://twitter.com/mattmatthews/status/927607712
  [Topic 3] [Image] https:// youtu.be/Xr4XdQ8K5a0?si=BgPqFjQ4j2Q4eBxWJ4b9dN1BgZFvQ1U9Sd1WzSjLm8PX9XjXuBZ-XJmG

  [VIX_LOW] [EPU_MED]
  [Topic 0] https://www. foxnews.com/politics/trump-straw mps-says-the-washington-post-is-wrong-about-bidens-law-prosecution-of-trump-and-haley-calls-la
  [Topic 1] The Judge in the Mueller Scam case is a phony, a fraud, and is in no way capable of protecting his Political Opponent, the Crooked New York 
  [Topic 2] https://www. washingtonexaminer.com/news/wa shington-secrets/trump-

In [15]:
import difflib

train_originals = df['gpt2_prompt'].astype(str).tolist()

def most_similar_training_tweet(generated_text, top_n=1):
    scores = [
        difflib.SequenceMatcher(None, generated_text.lower(), t.lower()).ratio()
        for t in train_originals
    ]
    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    return scores[best_idx], train_originals[best_idx]

print("Memorization check — similarity to nearest training tweet:\n")
flagged = 0
for _, row in samples_df.iterrows():
    score, match = most_similar_training_tweet(row['generated_tweet'])
    flag = " ⚠️  SUSPICIOUS" if score > 0.85 else ""
    print(f"[{row['vix_label']}][{row['epu_label']}][Topic {row['topic']}]  sim={score:.3f}{flag}")
    if score > 0.85:
        flagged += 1
        print(f"  Generated: {row['generated_tweet'][:100]}")
        print(f"  Nearest:   {match[:100]}\n")

print(f"\n{flagged}/{len(samples_df)} outputs flagged (>{0.85} similarity)")


Memorization check — similarity to nearest training tweet:

[VIX_LOW][EPU_LOW][Topic 0]  sim=0.465
[VIX_LOW][EPU_LOW][Topic 1]  sim=0.368
[VIX_LOW][EPU_LOW][Topic 2]  sim=0.435
[VIX_LOW][EPU_LOW][Topic 3]  sim=0.404
[VIX_LOW][EPU_MED][Topic 0]  sim=0.440
[VIX_LOW][EPU_MED][Topic 1]  sim=0.339
[VIX_LOW][EPU_MED][Topic 2]  sim=0.534
[VIX_LOW][EPU_MED][Topic 3]  sim=0.379
[VIX_LOW][EPU_HIGH][Topic 0]  sim=0.340
[VIX_LOW][EPU_HIGH][Topic 1]  sim=0.497
[VIX_LOW][EPU_HIGH][Topic 2]  sim=0.405
[VIX_LOW][EPU_HIGH][Topic 3]  sim=0.494
[VIX_MED][EPU_LOW][Topic 0]  sim=0.450
[VIX_MED][EPU_LOW][Topic 1]  sim=0.465
[VIX_MED][EPU_LOW][Topic 2]  sim=0.481
[VIX_MED][EPU_LOW][Topic 3]  sim=0.409
[VIX_MED][EPU_MED][Topic 0]  sim=0.501
[VIX_MED][EPU_MED][Topic 1]  sim=0.346
[VIX_MED][EPU_MED][Topic 2]  sim=0.454
[VIX_MED][EPU_MED][Topic 3]  sim=0.393
[VIX_MED][EPU_HIGH][Topic 0]  sim=0.330
[VIX_MED][EPU_HIGH][Topic 1]  sim=0.338
[VIX_MED][EPU_HIGH][Topic 2]  sim=0.366
[VIX_MED][EPU_HIGH][Topic 3]  sim=0.